[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/33.%20The%20Labor%20Shift%20Scheduling%20Problem/P33-Tier-3_executed.ipynb)

---


# 33. The Labor Shift Scheduling Problem

## Tier 3: A Genetic Algorithm

### Goal

This notebook implements a genetic algorithm (GA) for a weekly employee shift-scheduling problem. A GA maintains a population of candidate schedules and repeatedly applies selection, crossover, and mutation. Its fitness function rewards lower-cost schedules while penalizing unmet demand and violations of specified constraints.

The notebook's inputs are employee availability and weekly limits, shift costs, and daily demand. Its outputs are the best schedule found during the search, measures of schedule quality, and records of fitness and population diversity over generations. Because this is a stochastic search, a run does not guarantee an optimal schedule, and results can vary between runs.

### Model assumptions and scope

- Each assignment covers one employee on one day and one shift. The decision variable is binary: it is 1 when that assignment is made and 0 otherwise.
- Demand is a minimum requirement for each day and shift. Assigning more employees than required is allowed, but it does not earn additional coverage credit and still incurs assignment cost.
- The model penalizes, rather than forbids, multiple shifts in one day, exceeding an employee's weekly shift limit, and assigning an employee when unavailable. Consequently, a best-found schedule can still contain violations; the analysis reports them.
- Employee skills are stored in the scenario but are not connected to shift requirements. Breaks, rest periods between shifts, labor-law rules, preferences, and fairness objectives are outside this model.
- Availability is generated randomly when the scenario is created. No random seed is set, so rerunning the notebook can produce different availability and search results.

### The fitness model

Let $E$ be the set of employees, $D$ the set of days, and $S$ the set of shifts. The chromosome contains binary decisions $x_{eds}$ for employee $e$, day $d$, and shift $s$. The assignment cost is $c_{eds}$. Let $r_{ds}$ be required staffing and $a_{ds}=\sum_{e\in E}x_{eds}$ be assigned staffing. The shortage is $u_{ds}=\max(0,r_{ds}-a_{ds})$.

The algorithm maximizes the negative of a weighted penalty:

$$
f(x)=-\left(C(x)+1000U(x)+500V(x)\right),
\qquad
C(x)=\sum_{e\in E}\sum_{d\in D}\sum_{s\in S}c_{eds}x_{eds},
\qquad
U(x)=\sum_{d\in D}\sum_{s\in S}u_{ds}.
$$

Here, $V(x)$ is the sum of three violation measures: daily assignments in excess of one per employee, weekly assignments in excess of that employee's limit, and assignments made when the employee is unavailable. The constants 1000 and 500 are penalty weights, not probabilities. The fitness is larger (less negative) when the combined penalty is smaller. This is a penalty-based objective, not a guarantee that every constraint is satisfied.

### How the search proceeds

A chromosome is a three-dimensional binary array with shape $(|E|,|D|,|S|)$. The initial population is generated by assigning available employees to each demand slot when possible. Each generation evaluates fitness, selects parents by tournament selection, exchanges chromosome entries through uniform crossover, and flips eligible assignments through mutation. Elitism copies the best member of the current population into the next generation. The search records best, average, and worst fitness, along with mean pairwise Hamming distance as a measure of population diversity.

For population size $P$, chromosome length $N=|E||D||S|$, and $G$ generations, fitness evaluation is approximately $O(GPN)$. Computing all pairwise diversity values is approximately $O(GP^2N)$, so diversity tracking can be an important computational cost as the population grows. The algorithm is stochastic and does not provide an optimality certificate.

### Example configuration

The scenario contains 15 employees, 7 days, and 3 shift types. The main run uses a population of 50, at most 200 generations, a crossover rate of 0.8, and a mutation rate of 0.1. These are search settings, not promised outcomes. The actual fitness progression and schedule are computed when the notebook is run.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class Employee:
    """Employee attributes used by the scheduling scenario."""

    def __init__(self, employee_id, name, skills=None, max_shifts_per_week=5):
        self.id = employee_id
        self.name = name
        self.skills = skills or ["basic"]
        self.max_shifts_per_week = max_shifts_per_week
        self.availability = self._generate_availability()
        self.cost_multiplier = 1.0 + (employee_id - 1) * 0.05

    def _generate_availability(self):
        """Generate the scenario's random availability for every day and shift."""
        days = [
            "Monday", "Tuesday", "Wednesday", "Thursday",
            "Friday", "Saturday", "Sunday",
        ]
        shifts = ["Morning", "Afternoon", "Night"]
        availability = {}

        for day in days:
            for shift in shifts:
                base_probability = (
                    0.85
                    if day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
                    else 0.7
                )
                availability[(day, shift)] = random.random() < base_probability

        return availability

    def is_available(self, day, shift):
        return self.availability.get((day, shift), False)

    def get_cost(self, shift, day):
        base_costs = {"Morning": 100, "Afternoon": 120, "Night": 150}
        return base_costs.get(shift, 100) * self.cost_multiplier


class Shift:
    """A named shift type."""

    def __init__(self, shift_id, name):
        self.id = shift_id
        self.name = name


def create_ga_scenario():
    """Create the 15-employee, 7-day scheduling scenario."""
    employees = []
    for i in range(15):
        skills = ["basic"] if i % 3 != 0 else ["basic", "advanced"]
        max_shifts = 5 if i % 4 != 0 else 4
        employees.append(Employee(i + 1, f"E{i + 1}", skills, max_shifts))

    shifts = [
        Shift(1, "Morning"),
        Shift(2, "Afternoon"),
        Shift(3, "Night"),
    ]

    days = [
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday",
    ]

    demand = {
        "Monday": {"Morning": 4, "Afternoon": 5, "Night": 2},
        "Tuesday": {"Morning": 3, "Afternoon": 4, "Night": 2},
        "Wednesday": {"Morning": 4, "Afternoon": 5, "Night": 1},
        "Thursday": {"Morning": 3, "Afternoon": 4, "Night": 2},
        "Friday": {"Morning": 5, "Afternoon": 6, "Night": 3},
        "Saturday": {"Morning": 2, "Afternoon": 3, "Night": 2},
        "Sunday": {"Morning": 2, "Afternoon": 2, "Night": 1},
    }

    return employees, shifts, days, demand


employees, shifts, days, demand = create_ga_scenario()

print("GA Scenario Setup:")
print(f"Employees: {len(employees)}")
print(f"Shifts: {len(shifts)} ({', '.join(shift.name for shift in shifts)})")
print(f"Days: {len(days)} ({', '.join(days)})")

total_demand = sum(
    demand[day][shift.name]
    for day in days
    for shift in shifts
)
print("\nTotal Demand Requirements:")
print(f"  Total assignments required: {total_demand}")
for shift in shifts:
    shift_total = sum(demand[day][shift.name] for day in days)
    print(f"  {shift.name}: {shift_total} assignments")

## Genetic algorithm implementation

The chromosome is a candidate schedule: its entry `chromosome[e_idx, d_idx, s_idx]` corresponds to $x_{eds}$. A value of 1 means that the employee is assigned to that shift on that day. Fitness is evaluated over the whole chromosome, so decisions that are individually inexpensive can still be penalized if they leave demand uncovered or violate employee constraints.

Tournament selection samples candidates and chooses the one with the highest fitness. Uniform crossover independently exchanges entries between two parents. Mutation flips entries, but only adds an assignment when the employee is available. Crossover can still create unavailable assignments or other constraint violations; the fitness penalty measures these. The best chromosome from each generation is retained through elitism.

In [ ]:
class GeneticAlgorithmScheduler:
    """Genetic algorithm for the weekly shift-scheduling scenario."""

    def __init__(
        self,
        employees,
        shifts,
        days,
        demand,
        population_size=50,
        generations=200,
        crossover_rate=0.8,
        mutation_rate=0.1,
    ):
        self.employees = employees
        self.shifts = shifts
        self.days = days
        self.demand = demand
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate

        self.fitness_history = []
        self.diversity_history = []
        self.best_chromosome = None
        self.best_fitness = float("-inf")

        if self.population_size < 5:
            raise ValueError("Population size must be at least 5 for tournament selection.")
        if self.generations < 1:
            raise ValueError("The number of generations must be positive.")
        if not 0 <= self.crossover_rate <= 1:
            raise ValueError("Crossover rate must be between 0 and 1.")
        if not 0 <= self.mutation_rate <= 1:
            raise ValueError("Mutation rate must be between 0 and 1.")

        print(
            f"GA Initialized: population_size={population_size}, "
            f"generations={generations}"
        )
        print(
            f"Crossover rate: {crossover_rate}, "
            f"Mutation rate: {mutation_rate}"
        )

    def create_chromosome(self):
        """Create a random schedule, initially respecting availability."""
        chromosome = np.zeros(
            (len(self.employees), len(self.days), len(self.shifts)),
            dtype=int,
        )

        for d_idx, day in enumerate(self.days):
            for s_idx, shift in enumerate(self.shifts):
                required = self.demand[day][shift.name]
                available = [
                    e_idx
                    for e_idx, employee in enumerate(self.employees)
                    if employee.is_available(day, shift.name)
                ]

                # Assign as many available employees as the demand requires.
                selected = random.sample(available, min(required, len(available)))
                for employee_idx in selected:
                    chromosome[employee_idx, d_idx, s_idx] = 1

        return chromosome

    def calculate_fitness(self, chromosome):
        """Return the negative weighted penalty; higher fitness is better."""
        w_cost = 1.0
        w_coverage = 1000.0
        w_violation = 500.0

        expected_shape = (len(self.employees), len(self.days), len(self.shifts))
        if chromosome.shape != expected_shape:
            raise ValueError(f"Expected chromosome shape {expected_shape}.")

        cost_penalty = 0.0
        for e_idx, employee in enumerate(self.employees):
            for d_idx, day in enumerate(self.days):
                for s_idx, shift in enumerate(self.shifts):
                    if chromosome[e_idx, d_idx, s_idx] == 1:
                        cost_penalty += employee.get_cost(shift.name, day)

        coverage_penalty = 0.0
        for d_idx, day in enumerate(self.days):
            for s_idx, shift in enumerate(self.shifts):
                required = self.demand[day][shift.name]
                assigned = np.sum(chromosome[:, d_idx, s_idx])
                shortage = max(0, required - assigned)
                coverage_penalty += shortage * w_coverage

        violation_penalty = 0.0
        for e_idx, employee in enumerate(self.employees):
            for d_idx, day in enumerate(self.days):
                daily_shifts = np.sum(chromosome[e_idx, d_idx, :])
                if daily_shifts > 1:
                    violation_penalty += (daily_shifts - 1) * w_violation

            total_shifts = np.sum(chromosome[e_idx, :, :])
            if total_shifts > employee.max_shifts_per_week:
                excess = total_shifts - employee.max_shifts_per_week
                violation_penalty += excess * w_violation

            for d_idx, day in enumerate(self.days):
                for s_idx, shift in enumerate(self.shifts):
                    if chromosome[e_idx, d_idx, s_idx] == 1:
                        if not employee.is_available(day, shift.name):
                            violation_penalty += w_violation

        total_penalty = (
            w_cost * cost_penalty
            + coverage_penalty
            + violation_penalty
        )
        return -total_penalty

    def tournament_selection(self, population, fitness_scores, tournament_size=5):
        """Select the highest-fitness member of a random tournament."""
        indices = random.sample(range(len(population)), tournament_size)
        winner_idx = max(indices, key=lambda idx: fitness_scores[idx])
        return population[winner_idx].copy()

    def crossover(self, parent1, parent2):
        """Apply uniform crossover, or return copies of the parents."""
        if random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()

        child1 = parent1.copy()
        child2 = parent2.copy()
        mask = np.random.random(parent1.shape) < 0.5
        child1[mask] = parent2[mask]
        child2[mask] = parent1[mask]
        return child1, child2

    def mutate(self, chromosome):
        """Flip assignments probabilistically, respecting availability on additions."""
        mutated = chromosome.copy()

        for e_idx, employee in enumerate(self.employees):
            for d_idx, day in enumerate(self.days):
                for s_idx, shift in enumerate(self.shifts):
                    if random.random() < self.mutation_rate:
                        if mutated[e_idx, d_idx, s_idx] == 1:
                            mutated[e_idx, d_idx, s_idx] = 0
                        elif employee.is_available(day, shift.name):
                            mutated[e_idx, d_idx, s_idx] = 1

        return mutated

    def calculate_diversity(self, population):
        """Return mean pairwise Hamming distance between chromosomes."""
        if len(population) <= 1:
            return 0.0

        flattened = np.stack([chromosome.ravel() for chromosome in population])
        pairwise_distances = np.sum(
            flattened[:, np.newaxis, :] != flattened[np.newaxis, :, :],
            axis=2,
        )
        upper_triangle = np.triu_indices(len(population), k=1)
        return float(np.mean(pairwise_distances[upper_triangle]))

    def evolve(self):
        """Run selection, crossover, mutation, and elitist replacement."""
        print("\nStarting genetic algorithm evolution...")
        population = [
            self.create_chromosome()
            for _ in range(self.population_size)
        ]

        for generation in range(self.generations):
            fitness_scores = [
                self.calculate_fitness(chromosome)
                for chromosome in population
            ]

            current_best_idx = int(np.argmax(fitness_scores))
            current_best_fitness = fitness_scores[current_best_idx]
            if current_best_fitness > self.best_fitness:
                self.best_fitness = current_best_fitness
                self.best_chromosome = population[current_best_idx].copy()

            average_fitness = float(np.mean(fitness_scores))
            self.fitness_history.append(
                {
                    "generation": generation,
                    "best_fitness": current_best_fitness,
                    "avg_fitness": average_fitness,
                    "worst_fitness": min(fitness_scores),
                }
            )

            diversity = self.calculate_diversity(population)
            self.diversity_history.append(diversity)

            if generation % 25 == 0 or generation == self.generations - 1:
                print(
                    f"Generation {generation:3d}: "
                    f"Best={current_best_fitness:.0f}, "
                    f"Average={average_fitness:.0f}, "
                    f"Diversity={diversity:.1f}"
                )

            new_population = [population[current_best_idx].copy()]
            while len(new_population) < self.population_size:
                parent1 = self.tournament_selection(population, fitness_scores)
                parent2 = self.tournament_selection(population, fitness_scores)
                child1, child2 = self.crossover(parent1, parent2)
                new_population.append(self.mutate(child1))
                if len(new_population) < self.population_size:
                    new_population.append(self.mutate(child2))

            population = new_population

            if generation > 50 and len(self.fitness_history) >= 10:
                recent_best = [
                    record["best_fitness"]
                    for record in self.fitness_history[-10:]
                ]
                if np.allclose(recent_best, recent_best[0]):
                    print(f"Early stopping after no best-fitness change at generation {generation}.")
                    break

        print(f"\nEvolution completed. Best fitness: {self.best_fitness:.0f}")
        return self.best_chromosome, self.fitness_history, self.diversity_history

## Run the search

The population size, generation limit, crossover rate, and mutation rate control the search process. Elitism preserves the best candidate found in the current generation, while tournament selection and variation operators generate new candidates. The stopping condition can end the run early when the recorded best fitness has not changed across ten consecutive generations after the initial search period. It does not prove that the best candidate is globally optimal.

In [ ]:
ga_scheduler = GeneticAlgorithmScheduler(
    employees=employees,
    shifts=shifts,
    days=days,
    demand=demand,
    population_size=50,
    generations=200,
    crossover_rate=0.8,
    mutation_rate=0.1,
)

start_time = time.time()
best_chromosome, fitness_history, diversity_history = ga_scheduler.evolve()
execution_time = time.time() - start_time

print(f"\nGA execution time: {execution_time:.2f} seconds")

## Analyze the best-found schedule

The chromosome is converted into a readable schedule and checked against demand, availability, daily assignment limits, and weekly shift limits. Coverage is capped at 100% for each demand requirement: extra assignments do not count as additional fulfilled demand. The assignment count and cost still include every assignment, including any excess assignments.

In [ ]:
def analyze_ga_solution(chromosome, employees, shifts, days, demand):
    """Summarize assignments, demand coverage, cost, and constraint violations."""
    expected_shape = (len(employees), len(days), len(shifts))
    if chromosome.shape != expected_shape:
        raise ValueError(f"Expected chromosome shape {expected_shape}.")
    if not np.isin(chromosome, [0, 1]).all():
        raise ValueError("Chromosome entries must be binary.")

    schedule = {
        employee.name: {day: [] for day in days}
        for employee in employees
    }
    total_cost = 0.0
    total_assignments = 0

    for e_idx, employee in enumerate(employees):
        for d_idx, day in enumerate(days):
            for s_idx, shift in enumerate(shifts):
                if chromosome[e_idx, d_idx, s_idx] == 1:
                    schedule[employee.name][day].append(shift.name)
                    total_cost += employee.get_cost(shift.name, day)
                    total_assignments += 1

    coverage_stats = {}
    total_required = 0
    total_fulfilled = 0
    for d_idx, day in enumerate(days):
        coverage_stats[day] = {}
        for s_idx, shift in enumerate(shifts):
            required = demand[day][shift.name]
            assigned = int(np.sum(chromosome[:, d_idx, s_idx]))
            fulfilled = min(assigned, required)
            coverage_rate = 100.0 * fulfilled / required if required > 0 else 100.0
            coverage_stats[day][shift.name] = {
                "required": required,
                "assigned": assigned,
                "fulfilled": fulfilled,
                "coverage_rate": coverage_rate,
            }
            total_required += required
            total_fulfilled += fulfilled

    overall_coverage = (
        100.0 * total_fulfilled / total_required
        if total_required > 0
        else 100.0
    )

    violations = []
    for e_idx, employee in enumerate(employees):
        for d_idx, day in enumerate(days):
            daily_shifts = int(np.sum(chromosome[e_idx, d_idx, :]))
            if daily_shifts > 1:
                violations.append(
                    f"{employee.name} has {daily_shifts} shifts on {day}"
                )

        weekly_shifts = int(np.sum(chromosome[e_idx, :, :]))
        if weekly_shifts > employee.max_shifts_per_week:
            violations.append(
                f"{employee.name} works {weekly_shifts} shifts "
                f"(maximum: {employee.max_shifts_per_week})"
            )

        for d_idx, day in enumerate(days):
            for s_idx, shift in enumerate(shifts):
                if chromosome[e_idx, d_idx, s_idx] == 1:
                    if not employee.is_available(day, shift.name):
                        violations.append(
                            f"{employee.name} is unavailable for "
                            f"{shift.name} on {day}"
                        )

    return {
        "schedule": schedule,
        "total_cost": total_cost,
        "total_assignments": total_assignments,
        "overall_coverage": overall_coverage,
        "coverage_stats": coverage_stats,
        "violations": violations,
        "avg_cost_per_shift": (
            total_cost / total_assignments if total_assignments > 0 else 0.0
        ),
    }


solution_analysis = analyze_ga_solution(
    best_chromosome, employees, shifts, days, demand
)

print("\n" + "=" * 60)
print("GENETIC ALGORITHM SOLUTION ANALYSIS")
print("=" * 60)
print(f"Total assignment cost: ${solution_analysis['total_cost']:.2f}")
print(f"Total assignments: {solution_analysis['total_assignments']}")
print(f"Overall demand coverage: {solution_analysis['overall_coverage']:.1f}%")
print(f"Average cost per assignment: ${solution_analysis['avg_cost_per_shift']:.2f}")
print(f"Reported constraint violations: {len(solution_analysis['violations'])}")

if solution_analysis["violations"]:
    print("\nConstraint violations:")
    for violation in solution_analysis["violations"][:5]:
        print(f"  - {violation}")
    if len(solution_analysis["violations"]) > 5:
        remaining = len(solution_analysis["violations"]) - 5
        print(f"  - {remaining} additional violation(s)")
else:
    print("The analyzed schedule has no reported constraint violations.")

## Display the schedule

Each table entry lists an employee's assignment for that day. Employees with no assignment are marked `Off`; if a candidate has multiple shifts on one day, all of them are shown so that a penalized violation is not hidden by the display.

In [ ]:
def display_ga_schedule(schedule, employees, days):
    """Create and print a readable employee-by-day schedule table."""
    abbreviations = {
        "Morning": "Mor",
        "Afternoon": "Aft",
        "Night": "Nig",
    }
    schedule_matrix = {}
    for employee in employees:
        schedule_matrix[employee.name] = {}
        for day in days:
            assigned_shifts = schedule[employee.name][day]
            schedule_matrix[employee.name][day] = (
                "+".join(abbreviations[shift] for shift in assigned_shifts)
                if assigned_shifts
                else "Off"
            )

    schedule_df = pd.DataFrame(schedule_matrix).T
    schedule_df.index.name = "Employee"
    schedule_df.columns.name = "Day"

    print("\nGA-OPTIMIZED SCHEDULE:")
    print("=" * 80)
    print(schedule_df.to_string())
    return schedule_df


ga_schedule_df = display_ga_schedule(
    solution_analysis["schedule"], employees, days
)

## Visualize search progress and schedule characteristics

The fitness plot shows whether the best and typical population members improve under the maximization convention. Mean pairwise Hamming distance measures how many assignment bits differ between candidate schedules on average; a declining value can indicate that the population is becoming more similar. The schedule heatmap uses a separate category for multiple shifts in one day, making that type of violation visible.

In [ ]:
def create_ga_visualizations(
    fitness_history,
    diversity_history,
    solution_analysis,
    ga_schedule_df,
    employees,
    shifts,
    days,
):
    """Plot fitness, diversity, coverage, cost, and schedule summaries."""
    if not fitness_history:
        raise ValueError("Fitness history is empty; run the genetic algorithm first.")

    fig = plt.figure(figsize=(20, 12))
    generations = [record["generation"] for record in fitness_history]
    best_fitness = [record["best_fitness"] for record in fitness_history]
    average_fitness = [record["avg_fitness"] for record in fitness_history]
    worst_fitness = [record["worst_fitness"] for record in fitness_history]

    ax1 = plt.subplot(2, 4, 1)
    ax1.plot(generations, best_fitness, "g-", label="Best", linewidth=2)
    ax1.plot(generations, average_fitness, "b-", label="Average", linewidth=2)
    ax1.plot(generations, worst_fitness, "r-", label="Worst", linewidth=1, alpha=0.7)
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Fitness (higher is better)")
    ax1.set_title("Fitness Evolution")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = plt.subplot(2, 4, 2)
    ax2.plot(range(len(diversity_history)), diversity_history, color="purple", linewidth=2)
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Mean Pairwise Hamming Distance")
    ax2.set_title("Population Diversity")
    ax2.grid(True, alpha=0.3)

    ax3 = plt.subplot(2, 4, 3)
    categories = ["Off", "Mor", "Aft", "Nig", "Multiple"]
    category_codes = {category: code for code, category in enumerate(categories)}
    abbreviations = {"Morning": "Mor", "Afternoon": "Aft", "Night": "Nig"}
    numeric_schedule = np.zeros(ga_schedule_df.shape, dtype=int)
    for row_idx, employee_name in enumerate(ga_schedule_df.index):
        for col_idx, day in enumerate(ga_schedule_df.columns):
            assigned = solution_analysis["schedule"][employee_name][day]
            if not assigned:
                category = "Off"
            elif len(assigned) == 1:
                category = abbreviations[assigned[0]]
            else:
                category = "Multiple"
            numeric_schedule[row_idx, col_idx] = category_codes[category]

    colormap = plt.get_cmap("viridis", len(categories))
    image = ax3.imshow(numeric_schedule, cmap=colormap, aspect="auto", vmin=-0.5, vmax=len(categories) - 0.5)
    ax3.set_xticks(range(len(days)))
    ax3.set_xticklabels(days, rotation=45, ha="right")
    ax3.set_yticks(range(len(employees)))
    ax3.set_yticklabels([employee.name for employee in employees])
    ax3.set_xlabel("Day")
    ax3.set_ylabel("Employee")
    ax3.set_title("Schedule Assignments")
    colorbar = fig.colorbar(image, ax=ax3, ticks=range(len(categories)))
    colorbar.ax.set_yticklabels(categories)
    colorbar.set_label("Assignment category")

    ax4 = plt.subplot(2, 4, 4)
    for shift in shifts:
        rates = [
            solution_analysis["coverage_stats"][day][shift.name]["coverage_rate"]
            for day in days
        ]
        ax4.plot(days, rates, marker="o", label=shift.name, linewidth=2, markersize=4)
    ax4.set_xlabel("Day")
    ax4.set_ylabel("Fulfilled demand (%)")
    ax4.set_title("Demand Coverage by Shift")
    ax4.set_ylim(0, 105)
    ax4.tick_params(axis="x", rotation=45)
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    ax5 = plt.subplot(2, 4, 5)
    cost_by_shift = {shift.name: 0.0 for shift in shifts}
    employee_by_name = {employee.name: employee for employee in employees}
    for employee_name, employee_schedule in solution_analysis["schedule"].items():
        employee = employee_by_name[employee_name]
        for day, assigned_shifts in employee_schedule.items():
            for shift_name in assigned_shifts:
                cost_by_shift[shift_name] += employee.get_cost(shift_name, day)

    shift_names = [shift.name for shift in shifts]
    costs = [cost_by_shift[name] for name in shift_names]
    bars = ax5.bar(shift_names, costs, color=["gold", "lightcoral", "lightblue"])
    ax5.set_ylabel("Assignment cost ($)")
    ax5.set_title("Cost by Shift Type")
    ax5.grid(axis="y", alpha=0.3)
    for bar, cost in zip(bars, costs):
        ax5.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${cost:.0f}",
            ha="center",
            va="bottom",
        )

    ax6 = plt.subplot(2, 4, 6)
    workload_counts = [
        sum(len(shifts_for_day) for shifts_for_day in solution_analysis["schedule"][employee.name].values())
        for employee in employees
    ]
    ax6.bar(
        [employee.name for employee in employees],
        workload_counts,
        color="lightgreen",
        edgecolor="black",
    )
    ax6.set_xlabel("Employee")
    ax6.set_ylabel("Assigned shifts")
    ax6.set_title("Workload Distribution")
    ax6.tick_params(axis="x", rotation=45)
    ax6.grid(axis="y", alpha=0.3)

    ax7 = plt.subplot(2, 4, 7)
    improvement_rate = []
    for previous, current in zip(best_fitness[:-1], best_fitness[1:]):
        rate = (
            100.0 * (current - previous) / abs(previous)
            if previous != 0
            else 0.0
        )
        improvement_rate.append(rate)
    ax7.plot(range(1, len(generations)), improvement_rate, color="orange", linewidth=1)
    ax7.axhline(y=0, color="black", linewidth=1, alpha=0.4)
    ax7.set_xlabel("Generation")
    ax7.set_ylabel("Change in best fitness (%)")
    ax7.set_title("Generation-to-Generation Change")
    ax7.grid(True, alpha=0.3)

    ax8 = plt.subplot(2, 4, 8)
    window_size = 10
    moving_average = [
        np.mean(best_fitness[max(0, idx - window_size + 1):idx + 1])
        for idx in range(len(best_fitness))
    ]
    ax8.plot(generations, best_fitness, "g-", label="Best fitness", alpha=0.7)
    ax8.plot(
        generations,
        moving_average,
        "r-",
        label=f"{window_size}-generation moving average",
    )
    ax8.set_xlabel("Generation")
    ax8.set_ylabel("Fitness")
    ax8.set_title("Fitness Trend")
    ax8.legend()
    ax8.grid(True, alpha=0.3)

    fig.suptitle(
        f"Genetic Algorithm Analysis (Assignment cost: ${solution_analysis['total_cost']:.0f})",
        fontsize=14,
    )
    fig.tight_layout()
    plt.show()


create_ga_visualizations(
    fitness_history,
    diversity_history,
    solution_analysis,
    ga_schedule_df,
    employees,
    shifts,
    days,
)

## Interpret the run

The run's cost, coverage, constraint violations, and fitness history are empirical results for the generated availability and stochastic search. They should not be treated as guaranteed values. This notebook does not contain measured outputs from other scheduling methods, so it does not report numerical comparisons against them. A cost comparison is meaningful only when methods use the same scenario, cost definition, coverage requirements, and feasibility criteria.

In [ ]:
def summarize_ga_run(solution_analysis, fitness_history, execution_time):
    """Report measured results from this run without assuming a benchmark."""
    initial_best = fitness_history[0]["best_fitness"]
    final_best = fitness_history[-1]["best_fitness"]

    convergence_generation = None
    for index in range(10, len(fitness_history)):
        recent = [
            record["best_fitness"]
            for record in fitness_history[index - 10:index]
        ]
        if np.allclose(recent, recent[0]):
            convergence_generation = index
            break

    print("\n" + "=" * 60)
    print("MEASURED RUN SUMMARY")
    print("=" * 60)
    print(f"Assignment cost: ${solution_analysis['total_cost']:.2f}")
    print(f"Fulfilled demand: {solution_analysis['overall_coverage']:.1f}%")
    print(f"Reported constraint violations: {len(solution_analysis['violations'])}")
    print(f"Elapsed search time: {execution_time:.2f} seconds")
    print(f"Generations recorded: {len(fitness_history)}")
    print(f"Initial best fitness: {initial_best:.2f}")
    print(f"Final best fitness: {final_best:.2f}")

    if convergence_generation is None:
        print("No ten-generation fitness plateau was detected by this report.")
    else:
        print(
            "A ten-generation best-fitness plateau was detected near "
            f"generation {convergence_generation}."
        )

    return {
        "assignment_cost": solution_analysis["total_cost"],
        "coverage": solution_analysis["overall_coverage"],
        "violation_count": len(solution_analysis["violations"]),
        "elapsed_seconds": execution_time,
        "generations_recorded": len(fitness_history),
        "initial_best_fitness": initial_best,
        "final_best_fitness": final_best,
        "plateau_generation": convergence_generation,
    }


run_summary = summarize_ga_run(
    solution_analysis, fitness_history, execution_time
)

## Parameter sensitivity analysis

Population size, generation limit, crossover rate, and mutation rate influence how the population explores candidate schedules. The configurations below are evaluated on the same employee, demand, and availability data, but the random search differs between runs. Their measured costs and coverage can therefore illustrate sensitivity for this scenario; they do not establish that one configuration is universally best. The analysis uses separate scheduler instances so each run has its own fitness and diversity history.

In [ ]:
def parameter_sensitivity_analysis(employees, shifts, days, demand):
    """Evaluate the specified GA configurations on one fixed scenario."""
    print("\n" + "=" * 60)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 60)

    test_configs = [
        {"pop_size": 30, "generations": 100, "cross_rate": 0.7, "mut_rate": 0.05},
        {"pop_size": 50, "generations": 200, "cross_rate": 0.8, "mut_rate": 0.1},
        {"pop_size": 70, "generations": 300, "cross_rate": 0.9, "mut_rate": 0.15},
        {"pop_size": 40, "generations": 150, "cross_rate": 0.6, "mut_rate": 0.2},
    ]
    results = []

    for index, config in enumerate(test_configs, start=1):
        print(
            f"\nTest {index}: population={config['pop_size']}, "
            f"generations={config['generations']}, "
            f"crossover={config['cross_rate']}, "
            f"mutation={config['mut_rate']}"
        )

        test_ga = GeneticAlgorithmScheduler(
            employees=employees,
            shifts=shifts,
            days=days,
            demand=demand,
            population_size=config["pop_size"],
            generations=config["generations"],
            crossover_rate=config["cross_rate"],
            mutation_rate=config["mut_rate"],
        )

        start_time = time.time()
        test_chromosome, test_fitness_history, _ = test_ga.evolve()
        test_time = time.time() - start_time
        test_solution = analyze_ga_solution(
            test_chromosome, employees, shifts, days, demand
        )

        result = {
            "config": config,
            "cost": test_solution["total_cost"],
            "coverage": test_solution["overall_coverage"],
            "time": test_time,
            "fitness": test_ga.best_fitness,
            "generations_recorded": len(test_fitness_history),
            "violations": len(test_solution["violations"]),
        }
        results.append(result)
        print(
            f"Measured result: cost=${result['cost']:.2f}, "
            f"coverage={result['coverage']:.1f}%, "
            f"violations={result['violations']}, "
            f"time={result['time']:.2f} seconds"
        )

    print("\nParameter Sensitivity Results:")
    print(
        f"{'Run':<8} {'Cost':<12} {'Coverage':<12} "
        f"{'Violations':<12} {'Time (s)':<12} {'Generations':<12}"
    )
    print("-" * 68)
    for index, result in enumerate(results, start=1):
        print(
            f"{index:<8} ${result['cost']:<11.2f} "
            f"{result['coverage']:<11.1f}% "
            f"{result['violations']:<12} "
            f"{result['time']:<12.2f} "
            f"{result['generations_recorded']:<12}"
        )

    return results


sensitivity_results = parameter_sensitivity_analysis(
    employees, shifts, days, demand
)

## Summary and key insights

This notebook represents a weekly schedule as a binary chromosome and searches the space of schedules with a genetic algorithm. Fitness combines assignment cost with weighted penalties for uncovered demand, multiple daily shifts, weekly-limit excess, and unavailable assignments. Selection favors higher-fitness schedules; crossover and mutation introduce new candidate schedules; elitism retains a strong candidate across generations.

Interpret the reported quantities together. A low assignment cost alone does not imply a useful schedule if demand remains uncovered. High coverage does not establish feasibility when violations remain. The analysis reports these measures separately so that the trade-offs represented by the penalty weights are visible. The fitness value is a weighted score, not a monetary cost or an optimality certificate.

The fitness history describes the best, average, and worst candidates in the sampled population. The diversity history is the mean pairwise Hamming distance, measured in differing assignment entries. A plateau or declining diversity can indicate that the search population has become stable or similar, but neither observation proves global optimality.

The search is stochastic, its outcome depends on the generated availability and random choices, and no fixed seed is set. The sensitivity experiment provides observations for the listed parameter configurations on the generated scenario; it is not a general ranking of settings. Numerical comparisons with other scheduling methods require their actual results under matching scenario and evaluation rules.

The method is useful for studying population-based search when the schedule space is large and the objective combines several penalties. Its main limitations in this example are the absence of an optimality guarantee, dependence on parameter choices and random variation, and the simplified labor model. Natural extensions include representing additional operational constraints and testing larger scenarios, while keeping the same genetic-algorithm search structure.